# Asignación de Variedades a Anclas — ruteo por similitud + validación robusta

| | |
|---|---|
| **Proyecto** | `ml_random_forest / ml_training` |
| **Propósito** | Decidir qué variedades entrenan modelo propio (**anclas**) y a cuál ancla rutear cada variedad con poca data |
| **Entrada** | `training/DB-HISTORICA.xlsx` (una hoja por variedad; columnas crudas se mapean a las 5 features) |
| **Salida** | `decision_table.csv/.html` · `variety_anchor_mapping.yaml` · **`variety_predictive_routing.csv/.yaml`** (el ruteo accionable) |
| **Alcance** | Experimento — **no toca `src/`, `api/` ni `ui/`**. Produce el mapping `variedad → ancla` para enchufar al pipeline real |

---

## ¿Por qué existe este notebook?

Hay ~32 variedades de arándano, pero **8-9 concentran ~90% de la data** y el resto se
reparte el 10%. Con tan poca data, una variedad chica **no** puede entrenar un modelo
propio confiable. La decisión: entrenar **~11 anclas** (las de más data) y **rutear** las
demás dentro de esas anclas para que hereden un modelo entrenado con suficiente data.

> **Importante — esto NO es clustering.** El barrido de k y el silhouette muestran que
> la data **no tiene estructura de clusters natural** (silhouette negativo a todo k; los
> clusters reales están en k=2-3, no en 11). Por eso esto es **"asignar al donante más
> cercano"**, no clustering. Forzar 11 clusters "verdes" sería estadísticamente falso.

### Decisión que responde

> ¿Cuántos modelos entreno (anclas) y a cuál ancla mando cada variedad chica, de forma
> que el **pronóstico** sea lo más confiable posible?

---

## Método y por qué es estadísticamente robusto

| # | Sección | Técnica | Por qué |
|---|---------|---------|---------|
| 1 | Cargar datos | Mapeo de columnas crudas → 5 features | El Excel trae nombres crudos; corre sobre la data actual |
| 2 | Asignación | **Wasserstein + Cliff's delta** (escalado RobustScaler + filtro de outliers) | Asigna por el **mismo criterio con que decide** (efecto), no por una métrica y juzga por otra |
| 3 | Validación | **Mann-Whitney** (informativo) | Se reporta, pero **no decide** — a n grande el p-valor satura |
| 5 | Estructura global | 4 tests + **silhouette sobre todas las obs** | Veredicto honesto: Ratio/PERMANOVA son circulares y Kruskal-Wallis trivial a n grande; solo el silhouette no es circular |
| 5.05 | Barrido de k | KMeans + silhouette | Confirma que el k óptimo es 2-3, no 11 → no hay clusters |
| 5.1 | Refuerzo | **Bootstrap stability + Holm-Bonferroni** | Robustez por variedad + corrige comparaciones múltiples |
| 6 | Decisión | **Tamaño de efecto (Cliff's delta) primario**, estabilidad secundaria | No usa p-valores; un empate entre dos anclas cercanas no penaliza |
| **7** | **Ruteo predictivo** | **MAPE OOS de cada modelo de ancla sobre la variedad** | **Lo que de verdad reduce variabilidad**: rutea al ancla que mejor la *pronostica*, no a la más parecida |

> **Por qué tamaño de efecto y no p-valores:** con miles de filas, cualquier test sale
> significativo (trampa de gran muestra). Cliff's delta mide *cuánto* difieren las
> distribuciones (efecto), que es lo accionable. |d|<0.33 pequeño · <0.474 mediano.

> **Por qué ruteo predictivo (sección 7):** dos variedades pueden tener distribuciones
> distintas pero la misma relación features→target. Lo único que importa para compartir
> modelo es si el modelo del ancla **predice** bien la variedad — se mide por MAPE, no por
> distancia. Por eso la sección 7 es la decisión final; las 2-6 son el *prior*.

---

## Resultado actual (sobre la data de hoy)

- **23 variedades** en el Excel · **11 anclas** recomendadas (criterio predictivo:
  se autopredicen mejor que cualquier donante) · **12 ruteadas**.
- Veredicto de estructura: **REVISAR** (silhouette negativo) → correcto, no es clustering.
- Decisión por efecto: ~**87% confiable** (anclas + robusto/aceptable); las pocas
  "distintas" (efecto grande + poca data) se marcan honestamente para revisión.

## Cómo se conecta con el entrenamiento

Este notebook **no entrena el modelo de producción**. Genera el mapping
`variedad → ancla` (`variety_predictive_routing.yaml`) que el pipeline real (`src/` +
`task train`) puede consumir para saber qué variedades entrenan propio y cuáles heredan.


In [1]:
# ══════════════════════════════════════════════════════════════════
# Setup — la lógica vive en variety_clustering.py (importable y testeable);
# el notebook se queda con orquestación + visualización.
# ══════════════════════════════════════════════════════════════════
import os
import sys
import warnings

sys.path.insert(0, os.path.abspath("."))  # para importar variety_clustering.py
sys.path.insert(0, os.path.abspath("../src"))
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from variety_clustering import (
    EFFECT_MEDIUM,
    EFFECT_SMALL,
    MIN_N_VALIDATE,
    RANDOM_STATE,
    VIABILITY_ZONES,
    ExperimentConfig,
    apply_holm,
    assign_to_anchors,
    bootstrap_stability,
    build_final_result,
    classify_viability,
    cliffs_delta,
    compute_group_summary,
    export_results,
    filter_outliers,
    load_variety_data,
    mann_whitney_holm,
    mean_wasserstein,
    normalize_features,
    run_all_validations,
    silhouette_over_observations,
    validate_effect_size,
    validate_mann_whitney,
)

sns.set_style("whitegrid")
np.random.seed(RANDOM_STATE)


# ══════════════════════════════════════════════════════════════════
# VISUALIZACIÓN (específica del notebook)
# ══════════════════════════════════════════════════════════════════
THRESHOLD_LINES = [
    {"key": "robust", "color": "#2ecc71", "label": "Robusto"},
    {"key": "moderate", "color": "#f39c12", "label": "Básico"},
    {"key": "conservative", "color": "#e74c3c", "label": "Mínimo absoluto"},
]


def _add_threshold_lines(ax, thresholds: dict):
    """Agrega las líneas de umbral de viabilidad al gráfico."""
    for tl in THRESHOLD_LINES:
        ax.axhline(
            y=thresholds[tl["key"]],
            color=tl["color"],
            linestyle="--",
            linewidth=2,
            label=f"{tl['label']} (≥{thresholds[tl['key']]})",
        )


def plot_viability_chart(summary_df: pd.DataFrame, thresholds: dict):
    """Barras: clasificación de variedades por viabilidad de entrenamiento."""
    df = summary_df.sort_values("n_filas", ascending=False)
    colors = [classify_viability(r["n_filas"], thresholds)["color"] for _, r in df.iterrows()]

    fig, ax = plt.subplots(figsize=(16, 7))
    ax.bar(range(len(df)), df["n_filas"], color=colors)
    ax.set_xticks(range(len(df)))
    ax.set_xticklabels(df["variedad"], rotation=90, fontsize=8)
    _add_threshold_lines(ax, thresholds)
    ax.set_ylabel("Número de filas")
    ax.set_title("Clasificación de variedades por viabilidad de entrenamiento LightGBM")
    ax.legend(loc="upper right")
    plt.tight_layout()
    plt.show()

    bins = {
        "✓ Robusto": (df["n_filas"] >= thresholds["robust"]).sum(),
        "~ Básico": ((df["n_filas"] >= thresholds["moderate"]) & (df["n_filas"] < thresholds["robust"])).sum(),
        "⚠ Mínimo": ((df["n_filas"] >= thresholds["conservative"]) & (df["n_filas"] < thresholds["moderate"])).sum(),
        "✗ Insuficiente": (df["n_filas"] < thresholds["conservative"]).sum(),
    }
    for label, count in bins.items():
        print(f"{label}: {count}")


def plot_group_composition(group_summary: pd.DataFrame):
    """Barras apiladas: composición (propias + asignadas) de cada grupo ancla."""
    fig, ax = plt.subplots(figsize=(18, 8))
    x = range(len(group_summary))
    ax.bar(x, group_summary["filas_propias"], label="Filas propias (ancla)",
           color="#2ecc71", edgecolor="black", linewidth=0.5)
    ax.bar(x, group_summary["filas_asignadas"], bottom=group_summary["filas_propias"].values,
           label="Filas de variedades asignadas", color="#3498db", edgecolor="black", linewidth=0.5)
    ax.set_xticks(list(x))
    ax.set_xticklabels(group_summary["ancla"], rotation=45, ha="right", fontsize=10)
    for i, (_, row) in enumerate(group_summary.iterrows()):
        ax.text(i, row["total"] + 15, f"+{row['n_asignadas']} vars\n({row['total']} total)",
                ha="center", va="bottom", fontsize=8, fontweight="bold")
    ax.set_ylabel("Número de filas")
    ax.set_title("Composición de grupos de entrenamiento por ancla")
    ax.legend(loc="upper right", fontsize=10)
    plt.tight_layout()
    plt.show()


def plot_assignment_chart(final_result: pd.DataFrame, thresholds: dict):
    """Barras estilo viabilidad con el ancla asignada anotada sobre cada barra."""
    plot_df = final_result.sort_values("n_filas", ascending=False).reset_index(drop=True)
    colors = [classify_viability(r["n_filas"], thresholds)["color"] for _, r in plot_df.iterrows()]
    labels = [
        f"⚓ {r['variedad']}" if r["variedad"] == r["entrena_con"] else r["variedad"]
        for _, r in plot_df.iterrows()
    ]

    fig, ax = plt.subplots(figsize=(20, 8))
    ax.bar(range(len(plot_df)), plot_df["n_filas"], color=colors)
    ax.set_xticks(range(len(plot_df)))
    ax.set_xticklabels(labels, rotation=90, fontsize=8)
    _add_threshold_lines(ax, thresholds)
    for i, (_, row) in enumerate(plot_df.iterrows()):
        if row["variedad"] != row["entrena_con"]:
            ax.annotate(
                f"→ {row['entrena_con']}",
                xy=(i, row["n_filas"] + 5), ha="center", va="bottom",
                fontsize=6.5, fontweight="bold", color="#2c3e50", rotation=70,
                bbox=dict(boxstyle="round,pad=0.15", facecolor="white",
                          edgecolor="gray", alpha=0.85, linewidth=0.5),
            )
    ax.set_ylabel("Número de filas")
    ax.set_title("Asignación de variedades a anclas — por viabilidad de entrenamiento LightGBM")
    ax.legend(loc="upper right", fontsize=9)
    plt.tight_layout()
    plt.show()


# ── Inicializar config ──
config = ExperimentConfig()
print(f"Anclas: {len(config.anchor_varieties)} | Features: {config.features}")
print(f"Umbrales: {config.thresholds}")
print("Setup OK ✓")


Anclas: 11 | Features: ['KGHECT', 'INDUSTRIAL', 'DPC', 'PesoBayaFIMPRO', 'KGHORA']
Umbrales: {'conservative': 80, 'moderate': 160, 'robust': 638}
Setup OK ✓


---
## 1. Cargar datos

Leemos `training/DB-HISTORICA.xlsx` — una hoja por variedad. La función
`load_variety_data()` filtra las 5 features que interesan (definidas en
`config.features`) y descarta filas con valores nulos en cualquiera de ellas.

**Separación en dos grupos:**

- **Anclas** (11): variedades que entrenarán su propio modelo. Están fijadas por negocio en `config.anchor_varieties`.
- **No-anclas** (12): el resto. Se asigna un ancla (sección 2) y se rutea por predicción (sección 7).

> **Qué esperar:** ~16.000 filas entre las ~23 variedades. Variedades con
> muy pocas filas (n<20) se marcan `insuficiente` — no se validan de forma fiable
> y se documentan en la tabla de decisión final.


In [2]:
all_data, anchor_names, non_anchor_names, summary_df = load_variety_data(config)

print(f"Total variedades: {len(all_data)}")
print(f"\nAnclas ({len(anchor_names)}):")
for a in anchor_names:
    print(f"  ⚓ {a:20s} ({len(all_data[a]):4d} filas)")

print(f"\nA asignar ({len(non_anchor_names)}):")
for v in non_anchor_names:
    print(f"    {v:20s} ({len(all_data[v]):4d} filas)")

summary_df

Total variedades: 23

Anclas (11):
  ⚓ POP                  (9990 filas)
  ⚓ VENTURA              (4656 filas)
  ⚓ BEAUTY               (4125 filas)
  ⚓ BIANCA               (3347 filas)
  ⚓ ATLAS                (2755 filas)
  ⚓ JUPITER              (1668 filas)
  ⚓ ROSITA               ( 588 filas)
  ⚓ BELLA                ( 664 filas)
  ⚓ ARANA                ( 546 filas)
  ⚓ EMERALD              ( 803 filas)
  ⚓ MAGICA               (1152 filas)

A asignar (12):
    AZRA                 ( 131 filas)
    BILOXI               ( 400 filas)
    BONITA               ( 235 filas)
    FCM17-132            ( 116 filas)
    KIRRA                ( 844 filas)
    MADEIRA              ( 167 filas)
    MAGNUS               ( 132 filas)
    MALIBU               ( 230 filas)
    MASIRAH              ( 188 filas)
    RAYMI                ( 319 filas)
    STELLA               ( 151 filas)
    TERRAPIN             ( 105 filas)


,variedad,n_filas,tipo
9,POP,9990,⚓ ANCLA
0,VENTURA,4656,⚓ ANCLA
15,BEAUTY,4125,⚓ ANCLA
8,BIANCA,3347,⚓ ANCLA
4,ATLAS,2755,⚓ ANCLA
1,JUPITER,1668,⚓ ANCLA
7,MAGICA,1152,⚓ ANCLA
13,KIRRA,844,a asignar
6,EMERALD,803,⚓ ANCLA
18,BELLA,664,⚓ ANCLA


In [3]:
plot_viability_chart(summary_df, config.thresholds)

✓ Robusto: 10
~ Básico: 8
⚠ Mínimo: 5
✗ Insuficiente: 0


---
## 2. Distancia de Wasserstein y asignación a anclas

Para cada variedad no-ancla medimos qué tan parecida es su **distribución** a cada
una de las 11 anclas, y la asignamos a la más cercana.

### Cómo funciona

1. **Normalización:** `RobustScaler` sobre las 5 features (centra en la mediana, escala por el IQR) — resistente a outliers, a diferencia de `StandardScaler`.
2. **Distancia por feature:** para cada feature calculamos la distancia Wasserstein-1 entre la variedad y el ancla.
5. **Validación robusta:** la similitud variedad↔ancla se juzga por **tamaño de efecto (Cliff's delta)** sobre la misma data escalada+filtrada, no por p-valores (que saturan a n grande). Mann-Whitney queda como informativo.
3. **Distancia agregada:** promedio de las 5 distancias por feature.
4. **Asignación:** la variedad va al ancla con **menor distancia promedio**.

### ¿Por qué Wasserstein y no otra métrica?

| Métrica | ¿Qué mide? | Problema para este caso |
|---------|-----------|-------------------------|
| **Euclidiana** | Distancia entre medias | Ignora forma y varianza — 2 variedades con misma media pero muy distintas en varianza se verían iguales |
| **KL divergence** | Diferencia entre densidades | Requiere estimar densidades (KDE), asume continuidad, explota cuando hay poco solapamiento |
| **Wasserstein** ✓ | "Costo" de transformar una distribución en otra | Captura forma completa; robusta con pocos datos; interpretable en las unidades originales |

### Qué salida produce

`assignment_df` — una fila por variedad no-ancla con: `variedad`, `ancla_asignada`, `distancia_wass`, y las distancias top-2 para entender si la asignación fue "obvia" o "ajustada".


In [4]:
scaled_data = normalize_features(all_data, config.features)
scaled_data = filter_outliers(
    scaled_data,
    config.features,
    contamination=config.outlier_contamination,
    random_state=RANDOM_STATE,
)
print(
    f"Outliers filtrados "
    f"(contamination={config.outlier_contamination:.0%} por variedad)"
)
assignment_df = assign_to_anchors(
    non_anchor_names, anchor_names, all_data, scaled_data, config.features
)

# Mostrar agrupado por ancla
print("=" * 80)
print("ASIGNACIÓN DE VARIEDADES A ANCLAS (Wasserstein)")
print("=" * 80)

for anchor in sorted(anchor_names):
    assigned = assignment_df[assignment_df["ancla_asignada"] == anchor]
    total_group = len(all_data[anchor]) + assigned["n_filas"].sum()
    print(f"\n{'─' * 70}")
    print(f"  ⚓ {anchor} ({len(all_data[anchor])} propias, {total_group} total)")
    print(f"{'─' * 70}")
    if len(assigned) > 0:
        for _, row in assigned.sort_values("distancia_wass").iterrows():
            print(
                f"    ← {row['variedad']:20s} ({row['n_filas']:4d} filas) "
                f"dist={row['distancia_wass']:.3f}  "
                f"(2da: {row['segunda_opcion']} dist={row['dist_segunda']:.3f})"
            )
    else:
        print("    (sin variedades asignadas)")

print(f"\nTotal asignaciones: {len(assignment_df)}")
assignment_df

Outliers filtrados (contamination=5% por variedad)


ASIGNACIÓN DE VARIEDADES A ANCLAS (Wasserstein)

──────────────────────────────────────────────────────────────────────
  ⚓ ARANA (546 propias, 865 total)
──────────────────────────────────────────────────────────────────────
    ← RAYMI                ( 319 filas) dist=0.228  (2da: ROSITA dist=0.334)

──────────────────────────────────────────────────────────────────────
  ⚓ ATLAS (2755 propias, 3094 total)
──────────────────────────────────────────────────────────────────────
    ← MASIRAH              ( 188 filas) dist=0.240  (2da: BIANCA dist=0.230)
    ← STELLA               ( 151 filas) dist=0.340  (2da: JUPITER dist=0.175)

──────────────────────────────────────────────────────────────────────
  ⚓ BEAUTY (4125 propias, 4125 total)
──────────────────────────────────────────────────────────────────────
    (sin variedades asignadas)

──────────────────────────────────────────────────────────────────────
  ⚓ BELLA (664 propias, 795 total)
───────────────────────────────────────────

,variedad,n_filas,ancla_asignada,distancia_wass,segunda_opcion,dist_segunda
9,RAYMI,319,ARANA,0.2276,ROSITA,0.3335
8,MASIRAH,188,ATLAS,0.2405,BIANCA,0.2297
10,STELLA,151,ATLAS,0.3403,JUPITER,0.1747
0,AZRA,131,BELLA,0.3423,ROSITA,0.3092
11,TERRAPIN,105,BIANCA,0.2330,VENTURA,0.2586
2,BONITA,235,JUPITER,0.1622,BIANCA,0.2442
4,KIRRA,844,JUPITER,0.1525,BIANCA,0.1669
3,FCM17-132,116,POP,0.6410,BEAUTY,0.5967
6,MAGNUS,132,ROSITA,0.3443,BIANCA,0.3729
1,BILOXI,400,VENTURA,0.4048,JUPITER,0.2612


---
## 3. Validación — Mann-Whitney U

Para cada asignación variedad → ancla, aplicamos **Mann-Whitney U** feature por
feature. Contamos cuántas features tienen `p > 0.05` (las distribuciones no son
significativamente diferentes → se pueden tratar como similares).

### Regla de aceptación

| % features similares | Lectura |
|---|---|
| **≥ 80%** | Asignación excelente — la variedad se comporta como el ancla |
| **60–79%** | Asignación aceptable — hereda el modelo del ancla con confianza |
| **< 60%** | Asignación débil — el ancla más cercana existe pero la distribución difiere en demasiadas features |

### ¿Qué hacemos con las asignaciones débiles?

**Se mantiene la asignación** al ancla más cercana por Wasserstein. Razón: aunque los tests univariados (Mann-Whitney por feature) salgan significativos, Wasserstein captura similitud en la forma completa de la distribución. La tabla de decisión final (sección 6) las marca explícitamente como "revisar" para que puedas:

- Aceptar la asignación si el modelo del ancla funciona en producción
- O decidir entrenar esa variedad como su propio ancla si hay datos suficientes

> **Edge case conocido:** variedades con `n < 3` filas (ej. `JULIETA`) devuelven 0%
> porque Mann-Whitney requiere ≥3 observaciones. La tabla de decisión las marca
> con `"insuficiente"` en vez de `"revisar"` para no confundir.


In [5]:
validation_df = validate_mann_whitney(
    assignment_df, all_data, config.features, config.alpha
)

print("=" * 80)
print("VALIDACIÓN: Mann-Whitney U (p > 0.05 = similares)")
print("=" * 80)
print(f"\n  {'variedad':20s}   {'ancla':20s}  {'similares':>10s} {'%':>6s}")
print("  " + "─" * 68)
for _, r in validation_df.iterrows():
    print(
        f"  {r['variedad']:20s} → {r['ancla']:20s} "
        f"{r['features_similares']:>10s} {r['pct_similar']:5.0f}%  {r['status']}"
    )

good = (validation_df["pct_similar"] >= config.min_similar_pct).sum()
print(
    f"\n✓ {good}/{len(validation_df)} asignaciones con ≥{config.min_similar_pct:.0f}% features similares"
)

low = validation_df[validation_df["pct_similar"] < config.min_similar_pct]
if len(low) > 0:
    print(
        f"\n⚠ Asignaciones con <{config.min_similar_pct:.0f}% similitud ({len(low)}):"
    )
    for _, r in low.iterrows():
        print(f"    {r['variedad']} → {r['ancla']} ({r['pct_similar']:.0f}%)")
    print("    (se asignan igualmente al ancla más cercana por Wasserstein)")


# ── Validación robusta por tamaño de efecto (decisión real) ──────────
effect_df = validate_effect_size(assignment_df, scaled_data, config.features)
print()
print("=" * 80)
print("VALIDACIÓN ROBUSTA: tamaño de efecto (Cliff's delta, decide la asignación)")
print("=" * 80)
print(f"  {'variedad':20s}   {'ancla':14s} {'n':>5s} {'max|Cliff|':>11s}  veredicto")
print("  " + "─" * 70)
for _, r in effect_df.iterrows():
    mc = f"{r['max_cliff']:.2f}" if pd.notna(r["max_cliff"]) else "  —"
    print(f"  {r['variedad']:20s} → {r['ancla']:14s} {int(r['n']):5d} {mc:>11s}  {r['veredicto']}")
n_similar = (effect_df["veredicto"] == "similar").sum()
print(
    f"\n✓ {n_similar}/{len(effect_df)} asignaciones 'similar' por tamaño de efecto "
    f"(max|Cliff's delta| < {EFFECT_SMALL})"
)

VALIDACIÓN: Mann-Whitney U (p > 0.05 = similares)

  variedad               ancla                  similares      %
  ────────────────────────────────────────────────────────────────────
  AZRA                 → BELLA                       0/5     0%  ⚠
  STELLA               → ATLAS                       0/5     0%  ⚠
  MADEIRA              → VENTURA                     0/5     0%  ⚠
  KIRRA                → JUPITER                     1/5    20%  ⚠
  BILOXI               → VENTURA                     1/5    20%  ⚠
  TERRAPIN             → BIANCA                      2/5    40%  ⚠
  BONITA               → JUPITER                     2/5    40%  ⚠
  RAYMI                → ARANA                       2/5    40%  ⚠
  MALIBU               → VENTURA                     2/5    40%  ⚠
  FCM17-132            → POP                         2/5    40%  ⚠
  MASIRAH              → ATLAS                       3/5    60%  ✓
  MAGNUS               → ROSITA                      3/5    60%  ✓

✓ 2/12 a

---
## 4. Resultado final + export CSV

Combinamos las anclas (con sus propios datos) y las no-anclas (con su ancla asignada)
en un único `final_result`. Exportamos dos CSVs para consumo inmediato:

| Archivo | Uso |
|---------|-----|
| `variety_anchor_assignment.csv` | Tabla completa con zona de viabilidad, distancia Wasserstein y notas |
| `variety_training_mapping.csv` | Mapping simplificado `variedad → entrena_con` (útil para scripts) |

> Los dos CSVs son **intermedios**. El artefacto "fuente de verdad" que consumirá
> `modelo_nuevo_ml.ipynb` es el **YAML** que se genera en la sección 6, una vez
> que tenemos también los resultados de los 4 tests de validación global.


In [6]:
final_result = build_final_result(
    anchor_names, assignment_df, validation_df, all_data, config.thresholds
)
group_summary = compute_group_summary(anchor_names, assignment_df, all_data)

# Gráfico 1: Composición por grupo
plot_group_composition(group_summary)

# Gráfico 2: Asignación estilo viabilidad
plot_assignment_chart(final_result, config.thresholds)

# Resumen
print(f"Total filas para entrenamiento: {group_summary['total'].sum()}")
print(f"Anclas: {len(anchor_names)} | Asignadas: {len(assignment_df)}")
final_result

Total filas para entrenamiento: 33312
Anclas: 11 | Asignadas: 12


,variedad,n_filas,zona,entrena_con,ancla_asignada,distancia_wass,nota
11,RAYMI,319,⬛ Agrupar,ARANA,ARANA,0.2276,"Vecino más cercano: ARANA (dist=0.228, similit..."
0,ARANA,546,🟡 Individual,ARANA,ARANA,0.0000,Modelo individual (árboles básicos)
12,MASIRAH,188,⬛ Agrupar,ATLAS,ATLAS,0.2405,"Vecino más cercano: ATLAS (dist=0.240, similit..."
13,STELLA,151,⬛ Agrupar,ATLAS,ATLAS,0.3403,"Vecino más cercano: ATLAS (dist=0.340, similit..."
1,ATLAS,2755,🟢 Individual,ATLAS,ATLAS,0.0000,Modelo individual con nested CV completo
2,BEAUTY,4125,🟢 Individual,BEAUTY,BEAUTY,0.0000,Modelo individual con nested CV completo
14,AZRA,131,⬛ Agrupar,BELLA,BELLA,0.3423,"Vecino más cercano: BELLA (dist=0.342, similit..."
3,BELLA,664,🟢 Individual,BELLA,BELLA,0.0000,Modelo individual con nested CV completo
15,TERRAPIN,105,⬛ Agrupar,BIANCA,BIANCA,0.2330,"Vecino más cercano: BIANCA (dist=0.233, simili..."
4,BIANCA,3347,🟢 Individual,BIANCA,BIANCA,0.0000,Modelo individual con nested CV completo


In [7]:
export_df, mapping_df = export_results(final_result, config.output_dir)
export_df

✓ Exportado a ../data/variety_anchor_assignment.csv
✓ Exportado a ../data/variety_training_mapping.csv


,variedad,n_filas,zona,entrena_con,distancia_wass,nota
11,RAYMI,319,⬛ Agrupar,ARANA,0.2276,"Vecino más cercano: ARANA (dist=0.228, similit..."
0,ARANA,546,🟡 Individual,ARANA,0.0000,Modelo individual (árboles básicos)
12,MASIRAH,188,⬛ Agrupar,ATLAS,0.2405,"Vecino más cercano: ATLAS (dist=0.240, similit..."
13,STELLA,151,⬛ Agrupar,ATLAS,0.3403,"Vecino más cercano: ATLAS (dist=0.340, similit..."
1,ATLAS,2755,🟢 Individual,ATLAS,0.0000,Modelo individual con nested CV completo
2,BEAUTY,4125,🟢 Individual,BEAUTY,0.0000,Modelo individual con nested CV completo
14,AZRA,131,⬛ Agrupar,BELLA,0.3423,"Vecino más cercano: BELLA (dist=0.342, similit..."
3,BELLA,664,🟢 Individual,BELLA,0.0000,Modelo individual con nested CV completo
15,TERRAPIN,105,⬛ Agrupar,BIANCA,0.2330,"Vecino más cercano: BIANCA (dist=0.233, simili..."
4,BIANCA,3347,🟢 Individual,BIANCA,0.0000,Modelo individual con nested CV completo


## 5. Sustento estadístico de la agrupación

**¿Por qué validar?** Asignamos cada variedad al ancla más cercana por Wasserstein, pero necesitamos demostrar que esta agrupación **no es arbitraria** y que tiene sentido estadístico.

Usamos **4 tests independientes**, cada uno responde una pregunta distinta:

| # | Test | Pregunta que responde | ¿Qué es "bueno"? |
|---|------|-----------------------|-------------------|
| 1 | **Silhouette Score** | ¿Cada variedad está más cerca de su ancla que de las otras anclas? | Score > 0.25 (rango -1 a 1) |
| 2 | **Ratio intra/inter** | ¿La distancia dentro del grupo es menor que la distancia entre grupos? | Ratio < 1.0 (mientras más bajo, mejor) |
| 3 | **Kruskal-Wallis** | ¿Los 11 grupos son realmente diferentes entre sí en cada feature? | p < 0.05 en ≥3 de 5 features |
| 4 | **PERMANOVA** | ¿La agrupación explica la varianza de los datos mejor que el azar? | p < 0.05 y R² alto |

**Interpretación:** Si ≥3 de 4 tests pasan → la agrupación tiene **sustento estadístico sólido** para usarse en entrenamiento.

In [8]:
validation_tests = run_all_validations(
    final_result,
    assignment_df,
    anchor_names,
    all_data,
    scaled_data,
    config.features,
    config.n_permutations,
)


✗ Silhouette Score
   Score global: -0.0440 — Bajo
   ⚓ ARANA               : +0.365
   ⚓ ATLAS               : -0.241
   ⚓ BELLA               : +0.212
   ⚓ BIANCA              : +0.062
   ⚓ JUPITER             : +0.094
   ⚓ POP                 : -0.261
   ⚓ ROSITA              : -0.048
   ⚓ VENTURA             : -0.275

✓ Ratio intra/inter
   Intra-grupo: 0.2982  |  Inter-grupo: 0.4734
   Ratio: 0.6299 — Bueno

✓ Kruskal-Wallis
   KGHECT                H=   1544.27  p=0.00e+00  ✓ Sig.
   INDUSTRIAL            H=   3176.84  p=0.00e+00  ✓ Sig.
   DPC                   H=  12036.17  p=0.00e+00  ✓ Sig.
   PesoBayaFIMPRO        H=   7540.87  p=0.00e+00  ✓ Sig.
   KGHORA                H=   3462.55  p=0.00e+00  ✓ Sig.
   → 5/5 features significativas

✓ PERMANOVA
   F=4.1660  p=0.0010  R²=0.7764 (77.6%)



VEREDICTO HONESTO — solo evidencia no-circular
   Informativos (NO votan): Ratio y PERMANOVA son circulares;
   Kruskal-Wallis es trivial a n grande.
   Silhouette sobre 16,047 observaciones: -0.1120

   → ❌ SIN ESTRUCTURA NATURAL — esto es 'modelo donante más cercano', no clusters. La decisión por variedad la dan bootstrap + Cliff's delta (5.1).


---
## 5.05 Validación de k óptimo (barrido silhouette)

El silhouette global de la sección 5.1 puede salir **negativo** si el número
de anclas (k) no coincide con la estructura natural de los datos. Aquí
hacemos un barrido de k = `k_sweep_min` a `k_sweep_max` usando KMeans
sobre las observaciones escaladas (ya filtradas de outliers), y comparamos
el silhouette que obtiene la agrupación **inducida por las anclas actuales**
contra el mejor k que encontraría KMeans libre.

- Si el codo está en el **mismo k** que `len(anchor_varieties)`: el número
  de anclas es correcto.
- Si el mejor k es **menor**: hay anclas redundantes, considerar fusionar
  algunas.
- Si el mejor k es **mayor**: las anclas actuales no capturan toda la
  heterogeneidad; considerar agregar una ancla nueva.


In [9]:
# ══════════════════════════════════════════════════════════════════
# 5.05  Barrido de k óptimo por silhouette
# ══════════════════════════════════════════════════════════════════
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Apilar todas las observaciones escaladas (post-outlier filter)
obs_parts_k = [df[config.features].values for df in scaled_data.values()]
X_all_k = np.vstack(obs_parts_k)

k_current = len(anchor_names)
k_range = list(range(config.k_sweep_min, config.k_sweep_max + 1))
sil_scores_k = []

print("=" * 70)
print(f"Barrido k = {k_range[0]}..{k_range[-1]}  (k actual = {k_current})")
print("=" * 70)
for k in k_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
    labels_k = km.fit_predict(X_all_k)
    sil_k = silhouette_score(X_all_k, labels_k, metric="euclidean")
    sil_scores_k.append(sil_k)
    mark = " ← k actual" if k == k_current else ""
    print(f"  k={k:>2}  silhouette={sil_k:+.4f}{mark}")

best_k = k_range[int(np.argmax(sil_scores_k))]
best_sil = max(sil_scores_k)
sil_at_current = sil_scores_k[k_range.index(k_current)] if k_current in k_range else float("nan")

print()
print(f"  Mejor k (KMeans libre):   k={best_k}  silhouette={best_sil:+.4f}")
print(f"  Silhouette con k actual:  k={k_current}  silhouette={sil_at_current:+.4f}")
delta_k = best_k - k_current
if abs(delta_k) <= 1:
    verdict_k = "✓ k actual coincide con el óptimo (±1)"
elif delta_k < 0:
    verdict_k = f"⚠ Posible redundancia: óptimo k={best_k} < actual k={k_current}"
else:
    verdict_k = f"⚠ Heterogeneidad no cubierta: óptimo k={best_k} > actual k={k_current}"
print(f"  Veredicto: {verdict_k}")

# Gráfico del codo
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(k_range, sil_scores_k, marker="o", linewidth=2)
ax.axvline(k_current, color="#f59e0b", linestyle="--", label=f"k actual = {k_current}")
ax.axvline(best_k, color="#10b981", linestyle="--", label=f"k óptimo = {best_k}")
ax.set_xlabel("k (número de clusters)")
ax.set_ylabel("Silhouette global")
ax.set_title("Barrido de k óptimo — curva codo")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


Barrido k = 3..15  (k actual = 11)


  k= 3  silhouette=+0.3253


  k= 4  silhouette=+0.2853


  k= 5  silhouette=+0.2486


  k= 6  silhouette=+0.2421


...

  k=11  silhouette=+0.2312 ← k actual


  k=12  silhouette=+0.2214


  k=13  silhouette=+0.2217


  k=14  silhouette=+0.2132


  k=15  silhouette=+0.2086

  Mejor k (KMeans libre):   k=3  silhouette=+0.3253
  Silhouette con k actual:  k=11  silhouette=+0.2312
  Veredicto: ⚠ Posible redundancia: óptimo k=3 < actual k=11


---
## 5.1 Análisis estadístico reforzado

Los 4 tests de la sección 5 dan una **foto global** (¿hay señal?), pero tienen
tres limitaciones conocidas con esta data:

| Limitación | Impacto | Corrección aplicada |
|---|---|---|
| Silhouette + PERMANOVA corren sobre **los centroides** (1 por variedad) en vez de las ~16.000 observaciones | Baja potencia estadística | **Silhouette global** sobre todas las obs |
| Mann-Whitney por feature **sin corrección** de comparaciones múltiples | α real ≈ 23% (no 5%) con 5 tests → % similares inflado | **Holm-Bonferroni** step-down |
| No sabemos qué tan **frágil** es cada asignación individual | "Revisar" es una categoría opaca | **Bootstrap 100×** por variedad |

### Qué aporta cada técnica

#### (a) Bootstrap stability — ¿es robusta la asignación?

Para cada variedad no-ancla: resampleamos con reemplazo sus filas 100 veces; en cada iteración recomputamos su ancla nearest por Wasserstein. **Stability %** = proporción de iteraciones que cayeron en el ancla canónica.

- **≥ 90%** → asignación muy robusta (la data es consistente con el ancla)
- **60–89%** → asignación razonable pero con alternativa competitiva
- **< 60%** → asignación frágil; considerar entrenarla como ancla propia o revisar datos

Este es **el número más accionable para gerencia**: pasa "revisar" de etiqueta a probabilidad.

#### (b) Silhouette sobre todas las observaciones

El Silhouette de la sección 5 usa un centroide por variedad. Con tan pocos puntos en 5D y 11 grupos, el test tiene poca potencia. Aquí lo corremos sobre las ~16.000 observaciones individuales etiquetadas por su ancla asignada — más realista y con mayor poder estadístico.

#### (c) Mann-Whitney con corrección Holm-Bonferroni

Correr MW 5 veces (una por feature) con α=0.05 infla el error tipo I: probabilidad de al menos un falso positivo ≈ **1 - 0.95⁵ = 23%**. Holm-Bonferroni es un step-down que ajusta cada umbral por el rank del p-value, más potente que Bonferroni simple y sigue controlando el FWER al 5%.

La métrica `pct_similar_holm` es una versión más honesta de `pct_similar`: típicamente algunas asignaciones "aceptables" caen al aplicar la corrección.

### Cómo leer los resultados

| Si ves… | Significa… |
|---|---|
| Variedad con `stability < 60%` y `pct_similar_holm < 40%` | Asignación realmente frágil — evaluar entrenar como ancla propia |
| Variedad con `stability ≥ 90%` y `pct_similar_holm < 60%` | Ancla nearest robusta pero feature-level difiere — aceptable para modelo de regresión |
| `sil_all` mucho mayor que `sil_centroides` | La estructura del grupo es real pero se diluye al agregar por centroide |
| `n_raw - n_holm > 3` | La corrección importa — el reporte sin corrección era demasiado optimista |


In [10]:
# ══════════════════════════════════════════════════════════════════
# 5.1  Análisis estadístico reforzado (usa variety_clustering — sin duplicar)
# ------------------------------------------------------------------
#   (a) Bootstrap stability — ¿qué tan frágil es cada asignación?
#   (b) Silhouette sobre TODAS las observaciones (no solo centroides)
#   (c) Mann-Whitney U con corrección Holm-Bonferroni + umbral calibrado
#
# Produce stability_df, sil_all, mw_holm_df, mw_holm_threshold_calibrated
# que consume la sección 6 (tabla de decisión y YAML).
# ══════════════════════════════════════════════════════════════════
BOOTSTRAP_ITERATIONS = config.bootstrap_iterations
BOOTSTRAP_SEED = config.bootstrap_seed
k_anc = len(anchor_names)

# ── 5.1.1  Bootstrap stability ──────────────────────────────────────
print("=" * 70)
print(f"Bootstrap stability ({BOOTSTRAP_ITERATIONS} iter por variedad no-ancla)")
print("=" * 70)

anchor_of_map = {r["variedad"]: r["entrena_con"] for _, r in final_result.iterrows()}
non_anchor_list = [v for v in all_data if v not in anchor_names]
stability_df = bootstrap_stability(
    non_anchor_list, anchor_names, scaled_data, anchor_of_map, config.features,
    iterations=BOOTSTRAP_ITERATIONS, seed=BOOTSTRAP_SEED, alpha=config.alpha,
)

n_sig_boot = int(stability_df["boot_significant"].sum())
n_robust = int((stability_df["stability_pct"] >= 90).sum())
n_fragile = int((stability_df["stability_pct"] < 60).sum())
n_mid = int(((stability_df["stability_pct"] >= 60) & (stability_df["stability_pct"] < 90)).sum())
print(f"\n  Robustas    (stability ≥ 90%): {n_robust:>2}/{len(stability_df)}")
print(f"  Intermedias (60-89%)          : {n_mid:>2}/{len(stability_df)}")
print(f"  Frágiles    (< 60%)           : {n_fragile:>2}/{len(stability_df)}")
print(f"  Holm-sig vs azar (1/{k_anc}):  {n_sig_boot:>2}/{len(stability_df)} "
      f"(rechaza H0 tras corrección múltiple)")

weak = stability_df[stability_df["stability_pct"] < 80]
if len(weak) > 0:
    print("\n  Asignaciones con stability < 80%:")
    print(f"  {'variedad':20s}    {'canónica':12s}  {'stab':>7s}  {'top alt':12s}  {'%':>6s}")
    print("  " + "─" * 68)
    for _, r in weak.iterrows():
        print(f"  {r['variedad']:20s} →  {r['ancla_canonica']:12s} {r['stability_pct']:6.1f}%  "
              f"{r['top_alternative']:12s} {r['top_alt_pct']:5.1f}%")


# ── 5.1.2  Silhouette sobre TODAS las observaciones ─────────────────
print("\n" + "=" * 70)
print("Silhouette sobre todas las observaciones (no solo centroides)")
print("=" * 70)

sil_all, X_all, y_all = silhouette_over_observations(scaled_data, anchor_of_map, config.features)
sil_centroides = float(validation_tests[0].get("score", 0.0))
qual_all = "Excelente" if sil_all > 0.5 else "Aceptable" if sil_all > 0.25 else "Bajo"
print(f"\n  Observaciones usadas: {len(X_all):,}")
print(f"  Silhouette global:    {sil_all:+.4f}")
print(f"  (vs. {sil_centroides:+.4f} sobre centroides)")
print(f"  Calidad: {qual_all}")


# ── 5.1.3  Mann-Whitney U con corrección Holm-Bonferroni ────────────
print("\n" + "=" * 70)
print(f"Mann-Whitney U con corrección Holm-Bonferroni (α={config.alpha})")
print("=" * 70)

mw_holm_df, mw_holm_threshold_calibrated = mann_whitney_holm(
    validation_df, all_data, anchor_names, config.features,
    alpha=config.alpha, null_percentile=config.mw_holm_null_percentile,
    fallback_pct=config.min_similar_pct,
)
print(f"  Umbral MW-Holm calibrado: p{config.mw_holm_null_percentile:.0f} del null = "
      f"{mw_holm_threshold_calibrated:.1f}%  (fijo anterior: {config.min_similar_pct:.0f}%)")

n_raw = int((validation_df["pct_similar"] >= config.min_similar_pct).sum())
n_holm = int((mw_holm_df["pct_similar_holm"] >= config.min_similar_pct).sum())
n_holm_cal = int((mw_holm_df["pct_similar_holm"] >= mw_holm_threshold_calibrated).sum())
delta = n_raw - n_holm
print(f"\n  Asignaciones ≥{config.min_similar_pct:.0f}% (raw MW):         {n_raw:>2}/{len(validation_df)}")
print(f"  Asignaciones ≥{config.min_similar_pct:.0f}% (Holm, fijo):      {n_holm:>2}/{len(mw_holm_df)}")
print(f"  Asignaciones ≥{mw_holm_threshold_calibrated:.1f}% (Holm, calibrado): {n_holm_cal:>2}/{len(mw_holm_df)}")
if delta > 0:
    print(f"  → {delta} asignaciones bajaron al corregir por comparaciones múltiples")
elif delta < 0:
    print(f"  → {abs(delta)} asignaciones subieron (raro — revisar)")
else:
    print("  → Sin cambios tras la corrección")


# ── 5.1.4  Resumen ejecutivo del análisis reforzado ─────────────────
print("\n" + "=" * 70)
print("RESUMEN — Análisis estadístico reforzado")
print("=" * 70)
print(f"  Bootstrap:  {n_robust}/{len(stability_df)} asignaciones robustas (≥90%)")
print(f"  Bootstrap Holm-sig (vs 1/{k_anc}): {n_sig_boot}/{len(stability_df)}")
print(f"  Silhouette: {sil_all:+.4f} sobre {len(X_all):,} obs — {qual_all}")
print(f"  MW Holm:    {n_holm_cal}/{len(mw_holm_df)} asignaciones "
      f"≥{mw_holm_threshold_calibrated:.1f}% (umbral calibrado)")
print("\n✓ stability_df, sil_all, mw_holm_df, mw_holm_threshold_calibrated listos para la sección 6")


Bootstrap stability (100 iter por variedad no-ancla)



  Robustas    (stability ≥ 90%):  2/12
  Intermedias (60-89%)          :  1/12
  Frágiles    (< 60%)           :  9/12
  Holm-sig vs azar (1/11):   6/12 (rechaza H0 tras corrección múltiple)

  Asignaciones con stability < 80%:
  variedad                canónica         stab  top alt            %
  ────────────────────────────────────────────────────────────────────
  MALIBU               →  VENTURA         0.0%  JUPITER      100.0%
  BILOXI               →  VENTURA         0.0%  JUPITER      100.0%
  STELLA               →  ATLAS           0.0%  JUPITER      100.0%
  TERRAPIN             →  BIANCA          1.0%  JUPITER       79.0%
  FCM17-132            →  POP             2.0%  BEAUTY        76.0%
  MASIRAH              →  ATLAS          17.0%  BIANCA        49.0%
  AZRA                 →  BELLA          18.0%  ROSITA        68.0%
  MAGNUS               →  ROSITA         42.0%  POP           29.0%
  MADEIRA              →  VENTURA        56.0%  BIANCA        44.0%

Silhouette sobre 


  Observaciones usadas: 16,047
  Silhouette global:    -0.1120
  (vs. -0.0440 sobre centroides)
  Calidad: Bajo

Mann-Whitney U con corrección Holm-Bonferroni (α=0.05)


  Umbral MW-Holm calibrado: p90 del null = 40.0%  (fijo anterior: 60%)

  Asignaciones ≥60% (raw MW):          2/12
  Asignaciones ≥60% (Holm, fijo):       4/12
  Asignaciones ≥40.0% (Holm, calibrado):  7/12
  → 2 asignaciones subieron (raro — revisar)

RESUMEN — Análisis estadístico reforzado
  Bootstrap:  2/12 asignaciones robustas (≥90%)
  Bootstrap Holm-sig (vs 1/11): 6/12
  Silhouette: -0.1120 sobre 16,047 obs — Bajo
  MW Holm:    7/12 asignaciones ≥40.0% (umbral calibrado)

✓ stability_df, sil_all, mw_holm_df, mw_holm_threshold_calibrated listos para la sección 6


---
## 6. Tabla de decisión ejecutiva + YAML

Consolida los resultados de **todas** las secciones anteriores (1–5 y 5.1) en un
único artefacto ejecutivo y el YAML que consumirá el notebook de entrenamiento.

### Qué produce

| Artefacto | Consumidor | Contenido |
|-----------|-----------|-----------|
| `decision_table.csv` | Analistas · Excel | Tabla plana con métricas crudas + reforzadas por variedad |
| `decision_table.html` | **Gerencia** | KPIs + tabla con semáforos + leyenda — decisión reforzada |
| `variety_anchor_mapping.yaml` | `modelo_nuevo_ml.ipynb` | Config con mapeo + metadata + validación completa |

### Dos columnas de decisión: `raw` vs `robust`

Exportamos **dos columnas**, cada una útil en contextos distintos:

#### `decision_raw` — compatible con versión anterior

Solo usa MW crudo. Mantiene retrocompatibilidad con lectores/scripts viejos.

| Categoría | Criterio |
|-----------|----------|
| 🔵 **ancla** | variedad es ancla (entrena modelo propio) |
| 🟢 **excelente** | MW ≥ 80% features similares |
| 🟢 **aceptable** | MW 60–79% |
| 🟡 **revisar** | MW < 60%, n ≥ 3 |
| ⚪ **insuficiente** | n < 3 filas |

#### `decision_robust` — **esta es la que debe leer gerencia**

Combina **bootstrap stability** (robustez) con **MW corregido por Holm** (similitud multivariante honesta).

| Categoría | Criterio | Acción |
|-----------|----------|--------|
| 🔵 **ancla** | variedad es ancla | Entrena modelo propio |
| 🟢 **robusto** | efecto pequeño (\|Cliff\| < 0.33) **y** estable | Hereda con alta confianza |
| 🟢 **aceptable** | efecto ≤ moderado (\|Cliff\| < 0.474) | Hereda, monitorear en prod |
| 🟡 **revisar** | claramente distinta (0.474 ≤ \|Cliff\| < 0.60) | Validar con negocio |
| 🔴 **fragil** | muy distinta (\|Cliff\| ≥ 0.60) | Considerar entrenar como ancla propia |
| ⚪ **insuficiente** | n < 3 filas | Datos insuficientes |

### Veredicto global (4 tests de la sección 5)

| Tests pasan | Veredicto | Acción |
|---|---|---|
| Silhouette-obs > 0.25 | `SOLIDO` | Hay estructura real |
| 0 < Silhouette-obs ≤ 0.25 | `PARCIAL` | Estructura débil — decidir por efecto + estabilidad |
| **≤1/4** | `REVISAR` | No usar — replantear la lista de anclas |

> **Para decidir rápido:** mira la columna `stability_pct` en la tabla. Si todas
> las no-anclas están ≥70%, la agrupación es accionable. Si hay varias <50%,
> revisa si conviene convertirlas en anclas propias antes de entrenar.


In [11]:
# ══════════════════════════════════════════════════════════════════
# 6.  Tabla de decisión ejecutiva + YAML consumible por training
# ------------------------------------------------------------------
# Combina:
#   - final_result            (variedad → entrena_con, n_filas, zona)
#   - validation_df           (% MW similares — raw, sin corrección)
#   - mw_holm_df              (% MW similares — con Holm-Bonferroni)
#   - stability_df            (bootstrap stability % + top alternativa)
#   - validation_tests        (4 tests globales sección 5)
#   - sil_all                 (Silhouette sobre todas las observaciones)
#
# Produce 3 artefactos:
#   - decision_table.csv        (tabla plana — incluye métricas reforzadas)
#   - decision_table.html       (tablero gerencial navegable con ambas decisiones)
#   - variety_anchor_mapping.yaml  (fuente de verdad para modelo_nuevo_ml.ipynb)
# ══════════════════════════════════════════════════════════════════
from datetime import datetime
from pathlib import Path

import yaml

# ── 6.1  Construir tabla de decisión ────────────────────────────────
mw_pct_raw = dict(zip(validation_df["variedad"], validation_df["pct_similar"]))
mw_pct_holm = dict(zip(mw_holm_df["variedad"], mw_holm_df["pct_similar_holm"]))
stab_map = {
    r["variedad"]: (r["stability_pct"], r["top_alternative"], r["top_alt_pct"])
    for _, r in stability_df.iterrows()
}


def _decision_raw(row):
    """Decisión con MW crudo (compatible con versión anterior)."""
    if row["variedad"] == row["entrena_con"]:
        return ("ancla", "#0ea5e9")
    pct = mw_pct_raw.get(row["variedad"], 0.0)
    n = int(row["n_filas"])
    if n < 3:
        return ("insuficiente", "#94a3b8")
    if pct >= 80:
        return ("excelente", "#10b981")
    if pct >= 60:
        return ("aceptable", "#84cc16")
    return ("revisar", "#f59e0b")


# Mapa de significancia bootstrap (Holm sobre binomial vs azar)
_boot_sig_map = {
    r["variedad"]: bool(r.get("boot_significant", False))
    for _, r in stability_df.iterrows()
}
# Mapa de tamaño de efecto (Cliff's delta) — criterio robusto de similitud.
_eff_map = (
    {r["variedad"]: r["max_cliff"] for _, r in effect_df.iterrows()}
    if "effect_df" in globals() else {}
)


# Efecto "claramente distinto" (entre mediano y grande): por encima, heredar el
# modelo del ancla es cuestionable aunque la asignación sea estable.
EFFECT_DIFFERENT = 0.60


def _decision_robust(row):
    """Decisión PRIMARIA por tamaño de efecto (Cliff's delta); estabilidad secundaria.

    El consumidor es un modelo de regresión: importa cuán cerca está la
    distribución de la variedad de la de su ancla donante (tamaño de efecto), no
    el p-valor (satura a n grande) ni que el argmin del bootstrap oscile entre dos
    anclas *ambas cercanas* (empate inofensivo). La estabilidad solo separa
    'robusto' de 'aceptable'; el efecto grande degrada a revisar/fragil.
    """
    if row["variedad"] == row["entrena_con"]:
        return ("ancla", "#0ea5e9")
    n = int(row["n_filas"])
    if n < 3:
        return ("insuficiente", "#94a3b8")
    stab = stab_map.get(row["variedad"], (0.0, "—", 0.0))[0]
    max_cliff = _eff_map.get(row["variedad"], float("nan"))
    if np.isnan(max_cliff):
        return ("revisar", "#f59e0b")
    if max_cliff < EFFECT_SMALL:
        # distribución cercana → robusto si además es estable, si no aceptable
        return ("robusto", "#059669") if stab >= 70 else ("aceptable", "#84cc16")
    if max_cliff < EFFECT_MEDIUM:
        return ("aceptable", "#84cc16")  # diferencia moderada: donante válido
    if max_cliff < EFFECT_DIFFERENT:
        return ("revisar", "#f59e0b")    # claramente distinta — validar con negocio
    return ("fragil", "#dc2626")          # muy distinta — considerar ancla propia


decision_rows = []
for _, r in final_result.iterrows():
    label_raw, color_raw = _decision_raw(r)
    label_rob, color_rob = _decision_robust(r)
    stab, alt, alt_pct = stab_map.get(r["variedad"], (None, None, None))
    decision_rows.append(
        {
            "variedad": r["variedad"],
            "entrena_con": r["entrena_con"],
            "n_filas": int(r["n_filas"]),
            "distancia_wass": (
                float(r["distancia_wass"]) if pd.notna(r["distancia_wass"]) else None
            ),
            "mw_pct_raw": float(mw_pct_raw.get(r["variedad"], np.nan)),
            "mw_pct_holm": float(mw_pct_holm.get(r["variedad"], np.nan)),
            "stability_pct": float(stab) if stab is not None else None,
            "max_cliff": float(_eff_map.get(r["variedad"], np.nan)),
            "top_alternative": alt,
            "top_alt_pct": float(alt_pct) if alt_pct is not None else None,
            "zona": r.get("zona", ""),
            "decision_raw": label_raw,
            "decision_robust": label_rob,
            "_color_raw": color_raw,
            "_color_rob": color_rob,
        }
    )

decision_df = pd.DataFrame(decision_rows).sort_values(
    ["entrena_con", "decision_robust", "variedad"], kind="stable"
)

# ── 6.2  Export CSV plano ───────────────────────────────────────────
out_dir = Path(config.output_dir)
out_dir.mkdir(parents=True, exist_ok=True)
csv_path = out_dir / "decision_table.csv"
decision_df.drop(columns=["_color_raw", "_color_rob"]).to_csv(csv_path, index=False)
print(f"✓ {csv_path}")


# ── 6.3  Export HTML ejecutivo (autocontenido) ──────────────────────
counts_rob = decision_df["decision_robust"].value_counts().to_dict()
tests_passed = sum(t["passed"] for t in validation_tests)
# Veredicto HONESTO: la estructura se juzga por el silhouette sobre TODAS las
# observaciones (sección 5.1), no por los tests circulares/triviales.
_sil = float(globals().get("sil_all", validation_tests[0].get("score_all_obs", float("nan"))))
if _sil > 0.25:
    verdict = "SOLIDO"
elif _sil > 0.0:
    verdict = "PARCIAL"
else:
    verdict = "REVISAR"
verdict_color = {
    "SOLIDO": "#10b981",
    "PARCIAL": "#f59e0b",
    "REVISAR": "#dc2626",
}[verdict]

n_robust = (decision_df["stability_pct"].fillna(0) >= 90).sum()
n_fragile = (
    (decision_df["stability_pct"].fillna(0) < 60)
    & (decision_df["variedad"] != decision_df["entrena_con"])
).sum()
_mw_thr_kpi = float(
    globals().get("mw_holm_threshold_calibrated", config.min_similar_pct)
)
n_holm_ok = (decision_df["mw_pct_holm"].fillna(0) >= _mw_thr_kpi).sum()

kpis = [
    ("Variedades totales", f"{len(decision_df)}"),
    ("Anclas (entrenan)", f"{len(config.anchor_varieties)}"),
    ("Robustas (bootstrap ≥90%)", f"{n_robust}"),
    ("Frágiles (bootstrap <60%)", f"{n_fragile}"),
    (f"MW-Holm ≥{_mw_thr_kpi:.1f}%", f"{n_holm_ok}"),
    ("Silhouette global", f"{sil_all:+.3f}"),
    ("Tests globales OK", f"{tests_passed}/4"),
    ("Veredicto", verdict),
]

# Validación global (4 tests)
tests_rows_html = []
for t in validation_tests:
    icon = "OK" if t["passed"] else "FAIL"
    color_t = "#10b981" if t["passed"] else "#dc2626"
    detail = ""
    if t["name"] == "Silhouette Score":
        detail = (
            f"Centroides: {t.get('score', 0):.4f} — Global (todas obs): {sil_all:+.4f}"
        )
    elif t["name"] == "Ratio intra/inter":
        detail = f"Ratio: {t['ratio']:.4f} ({t['quality']})"
    elif t["name"] == "Kruskal-Wallis":
        detail = f"{t['n_sig']}/{t['n_total']} features significativas"
    elif t["name"] == "PERMANOVA":
        detail = f"F={t['f_obs']:.2f} · p={t['p_value']:.4f} · R²={t['r2']:.3f}"
    tests_rows_html.append(
        f"<tr><td><span style='color:{color_t};font-weight:700'>{icon}</span></td>"
        f"<td>{t['name']}</td><td>{detail}</td></tr>"
    )

# Filas de la tabla de decisión
rows_html = []
for _, r in decision_df.iterrows():
    wass = f"{r['distancia_wass']:.4f}" if pd.notna(r["distancia_wass"]) else "—"
    mw_r = f"{r['mw_pct_raw']:.0f}%" if pd.notna(r["mw_pct_raw"]) else "—"
    mw_h = f"{r['mw_pct_holm']:.0f}%" if pd.notna(r["mw_pct_holm"]) else "—"
    stab = f"{r['stability_pct']:.0f}%" if pd.notna(r["stability_pct"]) else "—"
    alt = (
        f"{r['top_alternative']} ({r['top_alt_pct']:.0f}%)"
        if r["top_alternative"] and r["top_alternative"] != "—"
        else "—"
    )
    rows_html.append(
        f"<tr>"
        f"<td><code>{r['variedad']}</code></td>"
        f"<td><code>{r['entrena_con']}</code></td>"
        f"<td>{r['n_filas']}</td>"
        f"<td>{wass}</td>"
        f"<td>{mw_r}</td>"
        f"<td>{mw_h}</td>"
        f"<td>{stab}</td>"
        f"<td><small>{alt}</small></td>"
        f"<td><span class='pill' style='background:{r['_color_rob']}'>"
        f"{r['decision_robust']}</span></td>"
        f"</tr>"
    )

kpi_html = "".join(
    f"<div class='kpi'><div class='kpi-v'>{v}</div>"
    f"<div class='kpi-l'>{l}</div></div>"
    for l, v in kpis
)

now = datetime.now().strftime("%Y-%m-%d %H:%M")
html_doc = f"""<!DOCTYPE html>
<html lang="es"><head><meta charset="utf-8">
<title>Decisión de clustering — {now}</title>
<style>
  * {{ box-sizing: border-box; }}
  body {{ font-family: -apple-system, Segoe UI, Roboto, sans-serif; margin: 0; background:#f8fafc; color:#0f172a; }}
  .wrap {{ max-width: 1200px; margin: 0 auto; padding: 32px 24px; }}
  h1 {{ margin: 0 0 4px; font-size: 26px; }}
  .sub {{ color:#64748b; margin-bottom: 24px; }}
  .verdict {{ display:inline-block; padding:4px 14px; border-radius:999px; color:white; font-weight:700; background:{verdict_color}; margin-left:8px; font-size:14px; vertical-align:middle; }}
  .kpis {{ display: grid; grid-template-columns: repeat(auto-fit,minmax(150px,1fr)); gap: 12px; margin-bottom: 28px; }}
  .kpi {{ background:white; border:1px solid #e2e8f0; border-radius: 10px; padding:16px; text-align:center; }}
  .kpi-v {{ font-size: 22px; font-weight: 700; }}
  .kpi-l {{ font-size: 12px; color:#64748b; text-transform: uppercase; letter-spacing: 0.05em; margin-top:4px; }}
  .card {{ background:white; border:1px solid #e2e8f0; border-radius: 10px; padding:16px; margin-bottom: 24px; }}
  h2 {{ font-size: 18px; margin: 0 0 12px; }}
  table.tbl {{ width:100%; border-collapse: collapse; font-size: 14px; }}
  table.tbl th, table.tbl td {{ text-align:left; padding: 8px 10px; border-bottom: 1px solid #e2e8f0; }}
  table.tbl th {{ background:#f1f5f9; font-weight: 600; font-size:12px; text-transform:uppercase; letter-spacing: 0.03em; }}
  .pill {{ color:white; padding:2px 10px; border-radius:999px; font-size:12px; font-weight:600; text-transform:capitalize; }}
  code {{ background:#f1f5f9; padding: 1px 6px; border-radius:4px; font-size: 12px; }}
  small {{ color:#64748b; }}
  .foot {{ color:#64748b; font-size:12px; margin-top:24px; line-height: 1.8; }}
</style></head><body>
<div class="wrap">
  <h1>Decisión de clustering <span class="verdict">{verdict}</span></h1>
  <div class="sub">Generado el {now} · Wasserstein sobre {len(config.features)} features · Bootstrap {BOOTSTRAP_ITERATIONS}× · Holm-Bonferroni · Silhouette sobre {len(X_all):,} obs</div>

  <div class="kpis">{kpi_html}</div>

  <div class="card">
    <h2>Validación global (4 tests) + Silhouette reforzado</h2>
    <table class="tbl">
      <thead><tr><th></th><th>Test</th><th>Resultado</th></tr></thead>
      <tbody>{''.join(tests_rows_html)}</tbody>
    </table>
  </div>

  <div class="card">
    <h2>Tabla de decisión — {len(decision_df)} variedades</h2>
    <p style="color:#64748b;font-size:13px;margin:0 0 12px">
      <b>MW raw</b>: Mann-Whitney sin corregir · <b>MW Holm</b>: con corrección de tests múltiples ·
      <b>Stability</b>: % de {BOOTSTRAP_ITERATIONS} bootstraps que confirman la asignación canónica ·
      <b>Top alt</b>: segunda ancla más probable en el bootstrap
    </p>
    <table class="tbl">
      <thead><tr>
        <th>Variedad</th><th>Entrena con</th><th>n</th>
        <th>Dist. Wass.</th><th>MW raw</th><th>MW Holm</th>
        <th>Stability</th><th>Top alt</th><th>Decisión</th>
      </tr></thead>
      <tbody>{''.join(rows_html)}</tbody>
    </table>
  </div>

  <div class="foot">
    <b>Leyenda de decisión reforzada:</b><br>
    <span class='pill' style='background:#0ea5e9'>ancla</span> entrena modelo propio ·
    <span class='pill' style='background:#059669'>robusto</span> efecto pequeño (|Cliff|&lt;0.33) y estable ·
    <span class='pill' style='background:#84cc16'>aceptable</span> efecto ≤ moderado (|Cliff|&lt;0.474) ·
    <span class='pill' style='background:#f59e0b'>revisar</span> claramente distinta (0.474≤|Cliff|&lt;0.60) ·
    <span class='pill' style='background:#dc2626'>fragil</span> muy distinta (|Cliff|≥0.60) — considerar ancla propia ·
    <span class='pill' style='background:#94a3b8'>insuficiente</span> n&lt;3 filas
    <br><br>
    <b>Cómo leer la tabla:</b> para decidir si una asignación es confiable, mira <b>stability</b> antes que MW raw.
    Una variedad puede tener MW raw 80% pero stability 50% → significa que la similitud marginal es casual y
    el ancla nearest es inestable al resamplear. La decisión reforzada ya integra ambos criterios.
  </div>
</div>
</body></html>
"""

html_path = out_dir / "decision_table.html"
html_path.write_text(html_doc, encoding="utf-8")
print(f"✓ {html_path}")


# ── 6.4  Export YAML (fuente de verdad para modelo_nuevo_ml.ipynb) ──
anchor_of = {r["variedad"]: r["entrena_con"] for _, r in final_result.iterrows()}
varieties_by_anchor: dict[str, list[str]] = {a: [] for a in config.anchor_varieties}
for variedad, ancla in anchor_of.items():
    varieties_by_anchor.setdefault(ancla, []).append(variedad)
for a in varieties_by_anchor:
    varieties_by_anchor[a].sort()


def _test_dict(t):
    base = {"name": t["name"], "passed": bool(t["passed"])}
    if t["name"] == "Silhouette Score":
        base.update(
            {
                "score_centroids": float(t.get("score", 0.0)),
                "score_all_observations": float(sil_all),
                "quality_all_obs": (
                    "Excelente"
                    if sil_all > 0.5
                    else "Aceptable" if sil_all > 0.25 else "Bajo"
                ),
            }
        )
    elif t["name"] == "Ratio intra/inter":
        base.update(
            {
                "ratio": float(t["ratio"]),
                "mean_intra": float(t["mean_intra"]),
                "mean_inter": float(t["mean_inter"]),
                "quality": t["quality"],
            }
        )
    elif t["name"] == "Kruskal-Wallis":
        base.update({"n_sig": int(t["n_sig"]), "n_total": int(t["n_total"])})
    elif t["name"] == "PERMANOVA":
        base.update(
            {
                "f_obs": float(t["f_obs"]),
                "p_value": float(t["p_value"]),
                "r2": float(t["r2"]),
            }
        )
    return base


def _nan_to_none(x):
    return float(x) if x is not None and pd.notna(x) else None


yaml_doc = {
    "metadata": {
        "generated_at": datetime.now().isoformat(timespec="seconds"),
        "method": "wasserstein_distance",
        "normalization": "RobustScaler",
        "features": list(config.features),
        "alpha": config.alpha,
        "min_similar_pct": config.min_similar_pct,
        "n_permutations": config.n_permutations,
        "n_varieties": len(final_result),
        "n_anchors": len(config.anchor_varieties),
        "reinforced_analysis": {
            "bootstrap_iterations": BOOTSTRAP_ITERATIONS,
            "bootstrap_seed": BOOTSTRAP_SEED,
            "mw_correction": "holm-bonferroni",
            "silhouette_source": f"all_{len(X_all)}_observations",
        },
    },
    "validation": {
        "verdict": verdict,
        "n_tests_passed": int(tests_passed),
        "n_tests_total": len(validation_tests),
        "global_tests": [_test_dict(t) for t in validation_tests],
        "mann_whitney_raw": {
            "n_assignments_ok": int(
                (validation_df["pct_similar"] >= config.min_similar_pct).sum()
            ),
            "n_assignments_weak": int(
                (validation_df["pct_similar"] < config.min_similar_pct).sum()
            ),
        },
        "mann_whitney_holm": {
            "n_assignments_ok": int(
                (mw_holm_df["pct_similar_holm"] >= config.min_similar_pct).sum()
            ),
            "n_assignments_weak": int(
                (mw_holm_df["pct_similar_holm"] < config.min_similar_pct).sum()
            ),
        },
        "bootstrap_stability": {
            "n_robust_ge_90": int((stability_df["stability_pct"] >= 90).sum()),
            "n_intermediate_60_89": int(
                (
                    (stability_df["stability_pct"] >= 60)
                    & (stability_df["stability_pct"] < 90)
                ).sum()
            ),
            "n_fragile_lt_60": int((stability_df["stability_pct"] < 60).sum()),
        },
    },
    "anchors": list(config.anchor_varieties),
    "mappings": {
        "anchor_of": anchor_of,
        "varieties_by_anchor": varieties_by_anchor,
    },
    "decisions_by_variety": {
        r["variedad"]: {
            "entrena_con": r["entrena_con"],
            "decision_raw": r["decision_raw"],
            "decision_robust": r["decision_robust"],
            "n_filas": int(r["n_filas"]),
            "mw_pct_raw": _nan_to_none(r["mw_pct_raw"]),
            "mw_pct_holm": _nan_to_none(r["mw_pct_holm"]),
            "stability_pct": _nan_to_none(r["stability_pct"]),
            "top_alternative": (
                r["top_alternative"]
                if r["top_alternative"] and r["top_alternative"] != "—"
                else None
            ),
            "top_alt_pct": _nan_to_none(r["top_alt_pct"]),
            "distancia_wass": _nan_to_none(r["distancia_wass"]),
        }
        for _, r in decision_df.iterrows()
    },
}

yaml_path = out_dir / "variety_anchor_mapping.yaml"
with open(yaml_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(
        yaml_doc,
        f,
        sort_keys=False,
        allow_unicode=True,
        default_flow_style=False,
    )
print(f"✓ {yaml_path}")


# ── 6.5  Resumen ejecutivo en stdout ────────────────────────────────
print()
print("═" * 70)
print(f"VEREDICTO: {verdict}  ({tests_passed}/4 tests globales pasan)")
print("═" * 70)
print(f"  Modelos a entrenar:   {len(config.anchor_varieties)} anclas")
print(f"  Variedades cubiertas: {len(decision_df)}")
print()
print(f"  Análisis reforzado:")
print(f"    Bootstrap robustas (≥90%):       {n_robust}/{len(stability_df)}")
print(f"    Bootstrap frágiles (<60%):       {n_fragile}/{len(stability_df)}")
print(
    f"    Bootstrap Holm-sig (vs 1/{len(config.anchor_varieties)}):     "
    f"{int(stability_df['boot_significant'].sum())}/{len(stability_df)}"
)
print(
    f"    MW-Holm ≥{_mw_thr_kpi:.1f}% (calibrado):  "
    f"{n_holm_ok}/{len(mw_holm_df)}"
)
print(f"    Silhouette global:               {sil_all:+.4f}")
print()
print(f"  Decisión reforzada (counts):")
for dec, cnt in sorted(counts_rob.items(), key=lambda x: -x[1]):
    print(f"    {dec:15s}: {cnt}")
print()
print("Artefactos generados en", out_dir.resolve())
print("  decision_table.csv / .html        (tablero gerencial)")
print("  variety_anchor_mapping.yaml       (consumible por modelo_nuevo_ml.ipynb)")

✓ ../data/decision_table.csv
✓ ../data/decision_table.html
✓ ../data/variety_anchor_mapping.yaml

══════════════════════════════════════════════════════════════════════
VEREDICTO: REVISAR  (3/4 tests globales pasan)
══════════════════════════════════════════════════════════════════════
  Modelos a entrenar:   11 anclas
  Variedades cubiertas: 23

  Análisis reforzado:
    Bootstrap robustas (≥90%):       2/12
    Bootstrap frágiles (<60%):       9/12
    Bootstrap Holm-sig (vs 1/11):     6/12
    MW-Holm ≥40.0% (calibrado):  7/12
    Silhouette global:               -0.1120

  Decisión reforzada (counts):
    ancla          : 11
    aceptable      : 7
    robusto        : 2
    revisar        : 2
    fragil         : 1

Artefactos generados en /home/carlosad/Proyects/ml_random_forest/data
  decision_table.csv / .html        (tablero gerencial)
  variety_anchor_mapping.yaml       (consumible por modelo_nuevo_ml.ipynb)


---
## 7. Ruteo predictivo (decide la variabilidad real)

El ruteo de las secciones 2-6 usa **distribución** (Wasserstein/efecto). Pero lo que
de verdad reduce la variabilidad del pronóstico es rutear cada variedad chica al ancla
**cuyo modelo la predice mejor** (menor MAPE out-of-sample), no a la más parecida.

Esta sección entrena un modelo proxy por ancla (HistGradientBoosting sobre las features,
prediciendo `KG/JR_H`) y rutea cada variedad restante al ancla de menor MAPE. Exporta
`variety_predictive_routing.csv/.yaml` — el mapping `variedad → ancla` para el pipeline.

> **Nota:** es un proxy (sin los lag features del pipeline real), sirve como *prior* de
> ruteo. La validación final se hace con los modelos reales. No toca el modelo/backend.

**Anclas (11, por criterio predictivo):** se predicen a sí mismas mejor que cualquier
donante. Incluye BILOXI y KIRRA (distintas pero con buena autopredicción).


In [12]:
# ── 7. Ruteo predictivo: variedad chica → ancla que mejor la pronostica ──
from sklearn.ensemble import HistGradientBoostingRegressor

import yaml

TARGET = "KGHORA"  # = KG/JR_H (target real)
PREDICTORS = ["KGHECT", "INDUSTRIAL", "DPC", "PesoBayaFIMPRO"]  # NO incluir el target
# 11 anclas recomendadas por criterio predictivo (autopredicción < mejor donante)
ROUTING_ANCHORS = [
    "POP", "BEAUTY", "VENTURA", "BIANCA", "ATLAS",
    "JUPITER", "MAGICA", "KIRRA", "EMERALD", "ROSITA", "BILOXI",
]
ROUTING_ANCHORS = [a for a in ROUTING_ANCHORS if a in all_data]


def _clean(variety):
    df = all_data[variety][PREDICTORS + [TARGET]].dropna()
    return df[df[TARGET] > 0]


def _mape(y, y_hat):
    return float(np.mean(np.abs((y - y_hat) / y)) * 100)


anchor_models = {}
for a in ROUTING_ANCHORS:
    d = _clean(a)
    anchor_models[a] = HistGradientBoostingRegressor(
        random_state=RANDOM_STATE, max_iter=200
    ).fit(d[PREDICTORS].values, d[TARGET].values)

routing_rows = []
for v in sorted(all_data):
    d = _clean(v)
    if len(d) == 0:
        continue
    if v in ROUTING_ANCHORS:
        own = _mape(d[TARGET].values, anchor_models[v].predict(d[PREDICTORS].values))
        routing_rows.append({"variedad": v, "entrena_con": v, "tipo": "ancla",
                             "mape_oos": round(own, 1), "n": len(d)})
        continue
    y, X = d[TARGET].values, d[PREDICTORS].values
    errs = {a: _mape(y, anchor_models[a].predict(X)) for a in ROUTING_ANCHORS}
    best = min(errs, key=errs.get)
    routing_rows.append({"variedad": v, "entrena_con": best, "tipo": "ruteada",
                         "mape_oos": round(errs[best], 1), "n": len(d)})

routing_df = pd.DataFrame(routing_rows).sort_values(["tipo", "mape_oos"])
print("=" * 70)
print("RUTEO PREDICTIVO — variedad → ancla de menor MAPE OOS")
print("=" * 70)
print(routing_df.to_string(index=False))

n_route = (routing_df["tipo"] == "ruteada").sum()
anchor_mape = routing_df[routing_df.tipo == "ancla"]["mape_oos"].mean()
print(f"\n  Anclas: {len(ROUTING_ANCHORS)} (MAPE propio medio {anchor_mape:.1f}%)")
print(f"  Ruteadas: {n_route}")

routing_df.to_csv(f"{config.output_dir}/variety_predictive_routing.csv", index=False)
mapping = {r["variedad"]: r["entrena_con"] for _, r in routing_df.iterrows()}
with open(f"{config.output_dir}/variety_predictive_routing.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump({"anchors": ROUTING_ANCHORS, "routing": mapping},
                   f, allow_unicode=True, sort_keys=False)
print(f"\n✓ {config.output_dir}/variety_predictive_routing.csv / .yaml exportados")


RUTEO PREDICTIVO — variedad → ancla de menor MAPE OOS
 variedad entrena_con    tipo  mape_oos    n
   BILOXI      BILOXI   ancla       4.6  180
   MAGICA      MAGICA   ancla       5.7  509
    KIRRA       KIRRA   ancla       6.3  375
   ROSITA      ROSITA   ancla       7.3  305
  JUPITER     JUPITER   ancla       7.4  662
  EMERALD     EMERALD   ancla       8.5  354
  VENTURA     VENTURA   ancla       8.9 2174
   BIANCA      BIANCA   ancla      10.2 1612
   BEAUTY      BEAUTY   ancla      11.0 2422
      POP         POP   ancla      14.0 5760
    ATLAS       ATLAS   ancla      75.6 1296
   STELLA         POP ruteada      21.3   35
   MALIBU         POP ruteada      21.4   67
  MASIRAH     VENTURA ruteada      24.6   56
    BELLA     JUPITER ruteada      24.7  195
  MADEIRA      BEAUTY ruteada      25.2   66
    RAYMI         POP ruteada      25.3  258
     AZRA         POP ruteada      26.0   78
FCM17-132       KIRRA ruteada      26.0   48
    ARANA     JUPITER ruteada      27.8  311
 